# HydroSeason - ERA5 Fetch Example

End-to-end workflow for catchments **without** an existing rainfall record:

1. Load a catchment polygon from a GeoJSON file
2. Fetch spatially-averaged monthly rainfall from the publicly accessible ERA5 Zarr archive on Google Cloud Storage (no API key required)
3. Optionally fetch additional ERA5 variables (temperature, evaporation)
4. Run the HydroSeason pipeline on the fetched data
5. Visualise and export results

> **Requirements:** this notebook installs the local checkout with `fetch` and `plot` extras.  
> An internet connection is required to stream the GCS Zarr store. Results are cached locally to avoid re-downloading.


In [ ]:
import sys
from pathlib import Path

_repo_root = Path("..").resolve()
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))


In [ ]:
# Install HydroSeason with ERA5 and plotting extras from this checkout.
%pip install -e "..[fetch,plot]" -q

## 1 — Imports


In [ ]:
import pandas as pd
import geopandas as gpd

from hydroseason.fetch import get_monthly_era5_rainfall
from hydroseason import classify_rainfall
from hydroseason.report import display_summary, generate_html_report
from hydroseason.plot import (
    plot_season_timeline,
    plot_monthly_climatology,
    plot_annual_metrics,
    plot_dashboard,
)


## 2 — Load the catchment polygon

The repository ships with `data/fitzroy_catchment.geojson` — an approximate bounding polygon for the Fitzroy River catchment in NW Australia — as a ready-to-run demonstration area.

Swap `GEOJSON_PATH` for any other polygon (Shapefile, GeoJSON, GeoPackage) to analyse a different catchment.


In [ ]:
GEOJSON_PATH = Path("../data/fitzroy_catchment.geojson")

gdf = gpd.read_file(GEOJSON_PATH)
print(f"CRS : {gdf.crs}")
print(f"Bounds: {gdf.total_bounds.round(4)}")
print(f"Name  : {gdf['name'].iloc[0]}")
gdf


## 3 — Fetch ERA5 monthly rainfall

`get_monthly_era5_rainfall` streams the publicly accessible ERA5 Zarr archive on Google Cloud Storage (**no API key or account required**), clips it to the catchment polygon, and returns a tidy monthly DataFrame with columns `[Date, Year, Month, Rainfall_mm]`.

Key parameters:
| Parameter | Description |
|---|---|
| `path` | GCS URI of the ERA5 Zarr store (fixed public path) |
| `gdf` | GeoDataFrame with the catchment polygon |
| `start_year` / `end_year` | Temporal range (inclusive) |
| `cache_dir` | Local folder to cache results — avoids re-downloading on subsequent runs |


In [ ]:
# ERA5 public Zarr store on Google Cloud Storage — no credentials required
ERA5_ZARR = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"

# Temporal range
START_YEAR = 1985
END_YEAR   = 2024

# Local cache: subsequent runs load from disk instead of re-streaming GCS
CACHE_DIR = Path("../data/era5_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

rainfall_df = get_monthly_era5_rainfall(
    path=ERA5_ZARR,
    gdf=gdf,
    start_year=START_YEAR,
    end_year=END_YEAR,
    variable="rainfall",
    cache_dir=CACHE_DIR,
    show_progress=True,
)

print(f"Fetched {len(rainfall_df)} monthly records  "
      f"({rainfall_df['Date'].min()} → {rainfall_df['Date'].max()})")
rainfall_df.head()


In [ ]:
out_csv = Path("../data/fitzroy_era5_rainfall.csv")
rainfall_df.to_csv(out_csv, index=False)
print(f"Saved → {out_csv.resolve()}")


## 4 — Run the HydroSeason pipeline

`classify_rainfall` accepts any DataFrame with columns `[Date, Year, Month, <value>]`.  
The `Rainfall_mm` column produced by `get_monthly_era5_rainfall` matches the default `value_col` directly.


In [ ]:
artifacts = classify_rainfall(rainfall_df)
result = artifacts.result

display_summary(artifacts)


## 5 — Visualise results


In [ ]:
# Season timeline — each month coloured by SeasonType, hydro-year boundaries marked
plot_season_timeline(result)


In [ ]:
# Monthly climatology — mean rainfall per calendar month coloured by baseline season
plot_monthly_climatology(result, artifacts.fixed_monthly)


In [ ]:
# Stacked wet/dry totals per hydrological year + wet month count
plot_annual_metrics(result)


In [ ]:
# Composite dashboard: timeline + climatology + annual totals in one figure
plot_dashboard(artifacts)


## 6 — Export results

Save the delineated season table as a CSV and generate a self-contained HTML report.


In [ ]:
# Season delineation table
result_csv = Path("../data/fitzroy_era5_hydroseason.csv")
result.to_csv(result_csv, index=False)
print(f"Results  → {result_csv.resolve()}")

# Self-contained HTML report (no Python needed to view)
report_path = generate_html_report(artifacts, "hydroseason_era5_report.html")
print(f"Report   → {report_path.resolve()}")
